# Chapter 4 — Introduction to Neural Learning: Gradient Descent

**Mata Kuliah:** Deep Learning
**Buku:** Grokking Deep Learning (Andrew W. Trask, Manning, 2019)
**Chapter 4:** Introduction to Neural Learning: Gradient Descent

Pada chapter sebelumnya saya sudah belajar cara membuat prediksi. Pertanyaan
berikutnya: bagaimana cara menentukan nilai weight agar prediksinya akurat?
Chapter ini menjawab dengan dua langkah lanjutan dari siklus *predict, compare,
learn*: **compare** (mengukur error) dan **learn** (memperbarui weight).

## Daftar Isi Pemahaman

1. Compare: mengukur error dengan mean squared error
2. Why measure error?
3. Hot and cold learning (mencoba naik dan turun)
4. Calculating direction and amount from error
5. Gradient descent untuk satu weight
6. Learning is just reducing error (hubungan error dan weight)
7. Derivative (turunan) dan cara memakainya untuk belajar
8. Divergence dan alpha

## 1. Compare: Does your network make good predictions?

setelah membuat prediksi, saya perlu tahu seberapa jauh
prediksi itu meleset. Ukuran ini disebut **error**. Di chapter ini digunakan
**mean squared error** dengan rumus `(pred - goal_pred) ** 2`.

Mengapa dikuadratkan? Analogi pemanah yang meleset 2 inci ke atas atau ke
bawah sama-sama meleset 2 inci. Kuadrat membuat error selalu positif, sehingga
tidak saling menghapus saat dirata-ratakan. Kuadrat juga membuat error besar
lebih diperhatikan daripada error kecil.

Contoh dengan `knob_weight = 0.5`, `input = 0.5`, dan `goal_pred = 0.8`
seharusnya menghasilkan error `0.3025`.

In [ ]:
knob_weight = 0.5
input = 0.5
goal_pred = 0.8

pred = input * knob_weight
error = (pred - goal_pred) ** 2
print(error)

Hasilnya `0.3025`. `goal_pred` adalah
nilai yang ingin diprediksi, sering kali sulit diamati langsung di dunia nyata.

## 2. Why measure error?

dengan mengubah error menjadi satu angka, masalah belajar
menjadi lebih sederhana. Tujuan saya bukan sekadar membuat `pred` mendekati
`goal_pred`, tetapi membuat `error` menjadi 0. Keduanya sebenarnya sama, tetapi
bentuk error lebih mudah dipikirkan.

Dengan mengkuadratkan selisih, error besar (misal 10 menjadi 100) dan error kecil
(misal 0,01 menjadi 0,0001) tidak diperlakukan sama. Error harus selalu positif agar
rata-rata error tidak menjadi 0 secara keliru. Misalnya dua error 1.000 dan -1.000
akan memberi rata-rata 0, padahal jaringan meleset jauh.

> Error hanya memberi tahu "meleset banyak" atau "meleset
> sedikit". Error tidak memberi tahu arah meleset atau cara memperbaikinya.
> Itu tugas langkah berikutnya, yaitu **learn**.

## 3. Hot and cold learning

cara paling sederhana untuk belajar adalah mencoba menaikkan
dan menurunkan weight, lalu memilih arah yang membuat error lebih kecil. Ini
disebut *hot and cold learning*.

Langkah-langkahnya:
1. Prediksi dengan weight saat ini dan hitung error.
2. Prediksi dengan `weight + step_amount` dan dengan `weight - step_amount`.
3. Pindahkan weight ke arah dengan error terkecil.
4. Ulangi sampai error mendekati 0.

Kode berikut mengikuti langkah tersebut dengan `goal_prediction = 0.8`.

In [ ]:
weight = 0.5
input = 0.5
goal_prediction = 0.8
step_amount = 0.001

for iteration in range(1101):
    prediction = input * weight
    error = (prediction - goal_prediction) ** 2
    print("Error:" + str(error) + " Prediction:" + str(prediction))

    # Coba naik
    up_prediction = input * (weight + step_amount)
    up_error = (goal_prediction - up_prediction) ** 2

    # Coba turun
    down_prediction = input * (weight - step_amount)
    down_error = (goal_prediction - down_prediction) ** 2

    if down_error < up_error:
        weight = weight - step_amount   # turun lebih baik
    if down_error > up_error:
        weight = weight + step_amount   # naik lebih baik

Pada iterasi ke-1101 prediksi mencapai `0.8`. Jika dilanjutkan,
prediksi berosilasi di sekitar `0.8`, karena langkahnya tetap.

Kelemahan hot and cold learning:
- **Tidak efisien.** Setiap pembaruan weight membutuhkan tiga prediksi.
- **`step_amount` tidak tepat.** Nilai langkah tidak bergantung pada besar error.
  Jika `step_amount` terlalu besar (misal 10), prediksi berosilasi dan tidak pernah
  mendekati `0.8`. Jika terlalu kecil, belajar lambat.

Idealnya ada cara menghitung **arah dan besar
perubahan** langsung dari error, tanpa perlu mencoba beberapa prediksi.

## 4. Calculating both direction and amount from error

rumus berikut menghitung arah dan besar perubahan dalam satu
baris, yaitu *gradient descent*:

`direction_and_amount = (pred - goal_pred) * input`

Bagian `(pred - goal_pred)` adalah **pure error**: arah dan besar kesalahan.
Mengalikannya dengan `input` melakukan tiga hal:
- **Stopping.** Jika `input = 0`, tidak ada yang dipelajari karena weight tidak
  memengaruhi prediksi.
- **Negative reversal.** Jika `input` negatif, arah weight dibalik, karena
  menaikkan weight justru menurunkan prediksi.
- **Scaling.** Input besar menghasilkan perubahan weight yang besar.

In [ ]:
weight = 0.5
goal_pred = 0.8
input = 0.5

for iteration in range(20):
    pred = input * weight
    error = (pred - goal_pred) ** 2
    direction_and_amount = (pred - goal_pred) * input
    weight = weight - direction_and_amount
    print("Error:" + str(error) + " Prediction:" + str(pred))

Output mendekati `0.8` secara bertahap. Error awal `0.3025` turun menjadi sekitar
`5e-06` setelah 20 iterasi.

## 5. One iteration of gradient descent (dengan node delta)

supaya lebih mudah dikembangkan ke jaringan yang lebih besar,
perhitungannya dipecah menjadi beberapa variabel:

- `delta = pred - goal_pred`: seberapa jauh **node output** meleset.
- `weight_delta = input * delta`: seberapa besar **weight** ikut menyebabkan
  kesalahan. Ini sama dengan `direction_and_amount` sebelumnya.
- `weight -= alpha * weight_delta`: perbarui weight dengan `alpha` sebagai
  pengendali kecepatan belajar.

Contoh: `number_of_toes = [8.5]`, `win_or_lose_binary = [1]`, dan `weight = 0.1`.
Dengan `alpha = 0.01`, weight naik sedikit, sama seperti hot and cold learning.

In [ ]:
weight, alpha = 0.1, 0.01

number_of_toes = [8.5]
win_or_lose_binary = [1]        # 1 = menang
input = number_of_toes[0]
goal_pred = win_or_lose_binary[0]

pred = input * weight                 # PREDICT
error = (pred - goal_pred) ** 2       # COMPARE
delta = pred - goal_pred              # node delta
weight_delta = input * delta          # weight delta
weight = weight - alpha * weight_delta  # LEARN

print('pred        :', pred)
print('error       :', error)
print('delta       :', delta)
print('weight_delta:', weight_delta)
print('weight baru :', weight)

Hasil: `pred = 0.85`, `error = 0.0225`, `delta = -0.15`, `weight_delta = -1.275`,
dan `weight` naik dari `0.1` menjadi `0.11275`. Karena `alpha = 0.01`, langkahnya kecil.
Perubahan weight ini searah dengan hot and cold learning, tetapi
besarnya langsung dihitung dari error.

## 6. Learning is just reducing error

jika digabungkan, `pred` dapat disubstitusikan ke rumus error:

`error = ((input * weight) - goal_pred) ** 2`

Dengan `input` dan `goal_pred` tetap, error hanya bergantung pada `weight`.
Grafiknya berbentuk parabola (kurva seperti mangkuk). Turunan (*slope*) pada setiap
titik menunjuk ke dasar mangkuk, yaitu error terkecil.

Contoh di bawah menjalankan gradient descent dengan `input = 1.1`. Perhatikan
weight bergerak melewati target lalu kembali mendekat.

In [ ]:
weight, goal_pred, input = (0.0, 0.8, 1.1)

for iteration in range(4):
    print("-----\nWeight:" + str(weight))
    pred = input * weight
    error = (pred - goal_pred) ** 2
    delta = pred - goal_pred
    weight_delta = delta * input
    weight = weight - weight_delta
    print("Error:" + str(error) + " Prediction:" + str(pred))
    print("Delta:" + str(delta) + " Weight Delta:" + str(weight_delta))

Output: error turun dari `0.64` menjadi sekitar `5e-05` dalam empat langkah.
Pada langkah pertama weight melompat terlalu jauh (`0.88`), lalu langkah berikutnya
mundur ke `0.6952`, dan terakhir mendekati `0.734`.

## 7. Derivative (turunan) dan cara memakainya

derivative adalah **sensitivitas** antara dua variabel: seberapa
banyak satu variabel berubah ketika variabel lain diubah. Contoh analogi batang
merah dan biru: jika `red_length = blue_length * 2`, derivative dari merah terhadap
biru adalah `2`. Jika derivative positif, keduanya bergerak searah; jika negatif,
berlawanan arah.

Untuk error, perlu diketahui bagaimana error berubah ketika weight berubah. Itu
derivative error terhadap weight. Karena error berbentuk kurva U, slope-nya:
- **Positif** di sebelah kanan dasar (weight perlu diturunkan).
- **Negatif** di sebelah kiri dasar (weight perlu dinaikkan).
- Makin jauh dari dasar, makin curam slope-nya.

Dengan demikian, arah pembaruan weight adalah **berlawanan dengan slope**. Dari
sinilah nama *gradient descent*: menuruni lereng seperti gravitasi.

Tidak perlu menghafal semua aturan kalkulus. Cukup tahu arti derivative dan
bahwa untuk fungsi error sederhana turunannya bisa dicari dari tabel. Pada
chapter ini, `weight_delta` adalah derivative error terhadap weight.

In [ ]:
# Kurva error untuk input = 0.5 dan goal_pred = 0.8
import numpy as np

input = 0.5
goal_pred = 0.8
weights = np.linspace(0, 2, 41)
errors = (input * weights - goal_pred) ** 2

for w, e in zip(weights[::8], errors[::8]):
    print(f'weight = {w:.2f}  error = {e:.4f}')

# Slope (derivative) error terhadap weight: 2 * input * (pred - goal_pred)
w = 0.5
slope = 2 * input * (input * w - goal_pred)
print(f'\nDi weight = 0.5, slope = {slope:.4f} (negatif -> weight harus dinaikkan)')

## 8. Divergence dan alpha

gradient descent bisa gagal. Jika `input` besar, derivative juga
besar, sehingga weight berubah terlalu banyak. Weight melewati dasar mangkuk dan
error justru membesar. Pola ini disebut **divergence**.

Dengan `input = 2` dan target `0.8`, tanpa `alpha` nilai prediksi meledak dan
bolak-balik antara positif dan negatif.

In [ ]:
weight = 0.5
goal_pred = 0.8
input = 2

for iteration in range(6):
    pred = input * weight
    error = (pred - goal_pred) ** 2
    derivative = input * (pred - goal_pred)
    weight = weight - derivative          # tanpa alpha
    print("Error:" + str(error) + " Prediction:" + str(pred))

Error membesar pada setiap langkah. Solusinya adalah mengalikan
pembaruan dengan **alpha**, bilangan antara 0 dan 1, sehingga pembaruan lebih
kecil:

`weight = weight - alpha * derivative`

Alpha tidak menghilangkan masalah input yang besar, tetapi membuat weight tidak
melompat terlalu jauh. Pemilihan alpha dilakukan dengan mencoba, misalnya
`10, 1, 0.1, 0.01, 0.001`, lalu memantau apakah error turun.

In [ ]:
weight = 0.5
goal_pred = 0.8
input = 2
alpha = 0.1

for iteration in range(20):
    pred = input * weight
    error = (pred - goal_pred) ** 2
    derivative = input * (pred - goal_pred)
    weight = weight - (alpha * derivative)
    print("Error:" + str(error) + " Prediction:" + str(pred))

Dengan `alpha = 0.1`, error turun dan prediksi mendekati `0.8`. Mengubah `alpha`:
- Jika terlalu besar, error kembali berosilasi atau membesar.
- Jika terlalu kecil, error turun sangat lambat.

## Latihan: menghafal kode gradient descent

Chapter ini menyarankan membangun kode gradient descent dari ingatan. Menulis ulang tanpa melihat catatan:

```python
weight, alpha = 0.5, 0.1
goal_pred, input = 0.8, 2

for _ in range(20):
    pred = input * weight
    delta = pred - goal_pred
    weight_delta = input * delta
    weight -= alpha * weight_delta
```

## Key Ideas — Chapter 4

- **Error** mengukur seberapa jauh prediksi meleset. Mean squared error selalu positif.
- **Hot and cold learning** mencoba dua arah, tetapi tidak efisien.
- **Gradient descent** menghitung arah dan besar perubahan langsung dari error dan input.
- **weight_delta** adalah derivative error terhadap weight.
- **Alpha** (learning rate) mencegah pembaruan terlalu besar dan mencegah divergence.
- Proses belajar adalah mengurangi error dengan mengubah weight.

## Kesimpulan

Chapter 4 menjelaskan bagaimana neural network belajar. Setelah prediksi dibuat,
error dihitung. Weight kemudian diperbarui ke arah yang mengurangi error, dengan
besar perubahan dihitung dari derivative. Alpha dipakai untuk menjaga agar pembaruan
tidak berlebihan. Konsep ini menjadi dasar untuk chapter berikutnya, yaitu
mempelajari banyak weight sekaligus.